# Install Dependencies

In [1]:
!pip install python-docx joblib --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 21.2 MB/s eta 0:00:00


# Imports

In [2]:
import os
import gc
import json
import numpy as np
import pandas as pd
from collections import Counter

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedShuffleSplit

import joblib
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from docx import Document
from datetime import datetime

# Mount Drive

In [3]:
from google.colab import drive
drive.mount('/users/')

Mounted at /users/


# Paths

In [4]:
RAW_BASE = "/users/"
BASE_SAVE_DIR = "/users/"

DATASET_NAME = "CIC_BCCC_NRC_IoMT_2024"
SAVE_DIR = os.path.join(BASE_SAVE_DIR, DATASET_NAME)
os.makedirs(SAVE_DIR, exist_ok=True)

print("📂 RAW:", RAW_BASE)
print("💾 SAVE:", SAVE_DIR)

📂 RAW: /users/
💾 SAVE: /users/


# Load All CSV Files & Merge

In [5]:
csv_files = [f for f in os.listdir(RAW_BASE) if f.endswith(".csv")]

dfs = []
for f in csv_files:
    path = os.path.join(RAW_BASE, f)
    class_name = f.replace(".csv", "").strip()

    print(f"Loading: {class_name}")
    df_temp = pd.read_csv(path, low_memory=False)

    # Attach textual label column from filename
    df_temp["Label"] = class_name
    dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True)
original_shape = df.shape

print("\n=== MERGED DATASET ===")
print("Shape:", original_shape)
df.head()

Loading: DDoS ICMP Flood
Loading: DoS ICMP Flood
Loading: DDoS UDP Flood
Loading: DoS TCP Flood
Loading: Benign Traffic
Loading: MITM ARP Spoofing
Loading: DoS UDP Flood
Loading: MQTT DDoS Publish Flood
Loading: MQTT Malformed
Loading: MQTT DoS Publish Flood
Loading: Recon OS Scan
Loading: MQTT DoS Connect Flood
Loading: Recon Port Scan
Loading: Recon Vulnerability Scan
Loading: Recon Ping Sweep

=== MERGED DATASET ===
Shape: (3385313, 85)


,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,Total Bwd packets,...,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Attack Name,Label
0,54.164.8.26-192.168.137.174-443-50634-6,54.164.8.26,443,192.168.137.174,50634,6,06/09/2023 02:00:42 PM,386988,25,42,...,0.000000,0.000000,0.0,0.0,0.00,0.000000,0.0,0.0,DDoS ICMP Flood,DDoS ICMP Flood
1,13.225.193.148-192.168.137.174-443-39086-6,13.225.193.148,443,192.168.137.174,39086,6,06/09/2023 02:00:51 PM,18583,3,0,...,0.000000,0.000000,0.0,0.0,0.00,0.000000,0.0,0.0,DDoS ICMP Flood,DDoS ICMP Flood
2,13.225.193.148-192.168.137.174-443-39086-6,13.225.193.148,443,192.168.137.174,39086,6,06/09/2023 02:00:51 PM,164448,1,3,...,0.000000,0.000000,0.0,0.0,0.00,0.000000,0.0,0.0,DDoS ICMP Flood,DDoS ICMP Flood
3,34.173.20.6-192.168.137.174-443-47304-6,34.173.20.6,443,192.168.137.174,47304,6,06/09/2023 02:00:36 PM,59949896,8,3,...,131709.333333,586.457444,132198.0,131059.0,14871682.75,108646.118192,14926518.0,14708716.0,DDoS ICMP Flood,DDoS ICMP Flood
4,192.168.137.174-34.173.20.6-47304-443-6,192.168.137.174,47304,34.173.20.6,443,6,06/09/2023 02:01:36 PM,1224,4,0,...,0.000000,0.000000,0.0,0.0,0.00,0.000000,0.0,0.0,DDoS ICMP Flood,DDoS ICMP Flood


# Encode Label (Text → Integer)

In [6]:
label_encoder = LabelEncoder()
df["Label_encoded"] = label_encoder.fit_transform(df["Label"])

classes = list(label_encoder.classes_)
num_classes = len(classes)

label_encoder_path = os.path.join(SAVE_DIR, "label_encoder.pkl")
joblib.dump(label_encoder, label_encoder_path)

print("✔ Classes:", classes)
print("💾 Saved label encoder:", label_encoder_path)

✔ Classes: ['Benign Traffic', 'DDoS ICMP Flood', 'DDoS UDP Flood', 'DoS ICMP Flood', 'DoS TCP Flood', 'DoS UDP Flood', 'MITM ARP Spoofing', 'MQTT DDoS Publish Flood', 'MQTT DoS Connect Flood', 'MQTT DoS Publish Flood', 'MQTT Malformed', 'Recon OS Scan', 'Recon Ping Sweep', 'Recon Port Scan', 'Recon Vulnerability Scan']
💾 Saved label encoder: /users/


# Select Numeric Features

In [7]:
X_raw = df.drop(columns=["Label", "Label_encoded"])
y = df["Label_encoded"].values

all_features = list(X_raw.columns)
numeric_cols = X_raw.select_dtypes(include=["number"]).columns.tolist()

X_num = X_raw[numeric_cols].copy()

print("Total Features:", len(all_features))
print("Numeric Features:", len(numeric_cols))

Total Features: 84
Numeric Features: 79


# Clean + Scale

In [8]:
X_num = X_num.replace([np.inf, -np.inf], np.nan)
X_num = X_num.fillna(X_num.median(numeric_only=True))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_num.values)

joblib.dump(scaler, os.path.join(SAVE_DIR, "scaler.pkl"))

print("✔ Scaled shape:", X_scaled.shape)

✔ Scaled shape: (3385313, 79)


# Train/Val/Test Split (70/15/15)

In [9]:
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(sss1.split(X_scaled, y))

X_train, X_temp = X_scaled[train_idx], X_scaled[temp_idx]
y_train, y_temp = y[train_idx], y[temp_idx]

sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(sss2.split(X_temp, y_temp))

X_val, X_test = X_temp[val_idx], X_temp[test_idx]
y_val, y_test = y_temp[val_idx], y_temp[test_idx]

print("Train:", X_train.shape)
print("Val  :", X_val.shape)
print("Test :", X_test.shape)

Train: (2369719, 79)
Val  : (507797, 79)
Test : (507797, 79)


# Class Distribution

In [10]:
def dist(arr):
    ct = Counter(arr.tolist())
    return {str(k): int(v) for k, v in ct.items()}

train_dist = dist(y_train)
val_dist = dist(y_val)
test_dist = dist(y_test)

print("Training Dist:", train_dist)
print("Validation Dist:", val_dist)
print("Test Dist:", test_dist)

Training Dist: {'7': 289739, '4': 1474841, '13': 339865, '8': 166622, '11': 59722, '14': 5825, '0': 22834, '2': 1803, '6': 737, '1': 1786, '5': 2181, '10': 1572, '9': 667, '3': 1475, '12': 50}
Validation Dist: {'4': 316038, '8': 35704, '7': 62087, '13': 72829, '14': 1248, '11': 12797, '0': 4893, '2': 387, '3': 316, '6': 158, '5': 467, '10': 337, '1': 383, '9': 143, '12': 10}
Test Dist: {'4': 316037, '8': 35705, '7': 62087, '11': 12798, '13': 72828, '0': 4893, '14': 1248, '3': 316, '1': 383, '5': 467, '10': 337, '9': 143, '6': 158, '2': 386, '12': 11}


# Autoencoder Model

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_dim = X_train.shape[1]
latent_dim = 64

In [12]:
class AE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        return out, z

In [13]:
ae = AE().to(device)
optimizer = torch.optim.Adam(ae.parameters(), lr=1e-3)
criterion = nn.MSELoss()

# Train Autoencoder

In [14]:
train_tensor = torch.tensor(X_train, dtype=torch.float32)
train_loader = DataLoader(TensorDataset(train_tensor), batch_size=1024, shuffle=True)

EPOCHS = 10
for epoch in range(1, EPOCHS + 1):
    ae.train()
    total_loss = 0
    for (batch,) in train_loader:
        batch = batch.to(device)

        optimizer.zero_grad()
        out, _ = ae(batch)
        loss = criterion(out, batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch}/{EPOCHS} - Loss: {total_loss/len(train_loader):.6f}")

Epoch 1/10 - Loss: 0.122117
Epoch 2/10 - Loss: 0.075266
Epoch 3/10 - Loss: 0.048258
Epoch 4/10 - Loss: 0.050160
Epoch 5/10 - Loss: 0.042060
Epoch 6/10 - Loss: 0.043673
Epoch 7/10 - Loss: 0.040864
Epoch 8/10 - Loss: 0.033028
Epoch 9/10 - Loss: 0.041387
Epoch 10/10 - Loss: 0.030357


# Latent Extraction

In [15]:
def encode(model, X):
    model.eval()
    with torch.no_grad():
        X_t = torch.tensor(X, dtype=torch.float32).to(device)
        _, Z = model(X_t)
    return Z.cpu().numpy()

z_train = encode(ae, X_train)
z_val   = encode(ae, X_val)
z_test  = encode(ae, X_test)

print(z_train.shape, z_val.shape, z_test.shape)

(2369719, 64) (507797, 64) (507797, 64)


# Save all .npy Files

In [16]:
np.save(os.path.join(SAVE_DIR, "train_latent.npy"), z_train)
np.save(os.path.join(SAVE_DIR, "val_latent.npy"),   z_val)
np.save(os.path.join(SAVE_DIR, "test_latent.npy"),  z_test)

np.save(os.path.join(SAVE_DIR, "y_train.npy"), y_train)
np.save(os.path.join(SAVE_DIR, "y_val.npy"),   y_val)
np.save(os.path.join(SAVE_DIR, "y_test.npy"),  y_test)

with open(os.path.join(SAVE_DIR, "feature_list.txt"), "w") as f:
    for c in numeric_cols:
        f.write(c + "\n")

torch.save(ae.state_dict(), os.path.join(SAVE_DIR, "autoencoder.pth"))

print("✔ ALL SAVED!")

✔ ALL SAVED!


# JSON Summary

In [17]:
summary = {
    "dataset_name": DATASET_NAME,
    "generated_at": datetime.now().isoformat(),
    "raw_shape": original_shape,
    "numeric_features": numeric_cols,
    "num_numeric_features": len(numeric_cols),
    "classes": {
        "num_classes": num_classes,
        "mapping": {int(i): str(cls) for i, cls in enumerate(classes)},
    },
    "splits": {
        "train": {"samples": len(y_train), "class_counts": train_dist},
        "val":   {"samples": len(y_val), "class_counts": val_dist},
        "test":  {"samples": len(y_test), "class_counts": test_dist},
    },
    "latent_dim": latent_dim,
}

json_path = os.path.join(SAVE_DIR, "preprocessing_summary.json")
with open(json_path, "w") as f:
    json.dump(summary, f, indent=4)

print("Saved:", json_path)

Saved: /users/


# DOCX Summary

In [18]:
doc = Document()
doc.add_heading(f"Dataset Preprocessing Summary — {DATASET_NAME}", level=1)

doc.add_paragraph(f"Generated at: {datetime.now()}")
doc.add_paragraph(f"Total rows: {original_shape[0]}")
doc.add_paragraph(f"Total columns: {original_shape[1]}")

doc.add_heading("Numeric Features", level=2)
doc.add_paragraph(f"Count: {len(numeric_cols)}")

doc.add_heading("Classes", level=2)
table = doc.add_table(rows=1, cols=2)
table.rows[0].cells[0].text = "Index"
table.rows[0].cells[1].text = "Class Name"

for i, cls in enumerate(classes):
    row = table.add_row().cells
    row[0].text = str(i)
    row[1].text = cls

doc_path = os.path.join(SAVE_DIR, "preprocessing_summary.docx")
doc.save(doc_path)

print("DOCX Saved:", doc_path)

DOCX Saved: /users/
